# BiLSTM soil-moisture forecast, 72h ahead (FD11): model_test setup

`model_test.ipynb`'s pipeline, run on the TxSON33 parquet with the data handling of `lstm_72h.ipynb`.

| | |
|---|---|
| From `model_test` | 48h input window; target = the **daily mean** of `SWC_20` (`predict_avg`); hour-of-day inputs (`DaySin`, `DayCos`); its BiLSTM (input projection, 2-layer BiLSTM with hidden size 64, LayerNorm, dropout 0.2, MLP head); batch size 32; stride 8 for training and validation windows; 30 epochs, patience 10; a whole-test-period plot |
| From `lstm_72h` | station `FD11`, 72h horizon; train 2019–2021, validation 2022, test 2024 (2023 skipped: the rain gauge recorded nothing from 2022-12-04 to 2023-09-24); 1 mm/h rain deadband and the engineered soil and rain features; scaler fit on the training years only; observed targets only; change target with Huber loss; Adam lr 1e-3, no weight decay; last-value and 24h moving-average baselines |

In [ ]:
import hashlib
import os
import random
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# Local path, or your Drive path. If the file is missing, Section 1 downloads it from the project's GitHub repo.
PARQUET_PATH = "TxSON33_modeling_complete.parquet"
PARQUET_URL = ("https://raw.githubusercontent.com/AkhilJar/tx-soil-moisture/a61427b2c1f45fb1a607f2b9aa2c73cc2aa32701/"
               "datasets/txson33_complete_for_modeling/TxSON33_modeling_complete.parquet")
PARQUET_SHA256 = "c7b71436bcd201e097fefc7f4a4efc20366d0642e2c16566dfc595b0f91af4c9"

STATION = "FD11"
TRAIN_YEARS, VAL_YEARS, TEST_YEARS = [2019, 2020, 2021], [2022], [2024]   # 2023 skipped (rain gauge outage)

# model_test parameters
WINDOW_SIZE = 48           # window_size: hours of input
HORIZON = 72               # offset: target hour = the window's last hour + 72 h (model_test used 48)
TRAINING_STRIDE = 8        # training_stride: keep every 8th training window
VALIDATION_STRIDE = 8      # validation_stride
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 10
# model_test used lr 1e-4 with weight decay 1e-2. On the change target that shrank the forecast to last value minus
# a near-constant amount (10/29 check: CORR identical to last value's), so this uses lstm_72h's settings.
LR = 1e-3
WEIGHT_DECAY = 0.0

# lstm_72h elements
PPT_DEADBAND_MM = 1.0      # hourly rain below 1 mm is set to 0 before every rain feature
HUBER_DELTA = 0.005        # Huber threshold in SWC_20 units
FEATURES = ["SWC_20", "SWC_20_change_24h", "SWC_20_change_72h", "SWC_20_below_max_72h", "SWC_20_above_min_30d",
            "SWC_5", "Ppt_rolling_24h_log", "Ppt_rolling_72h_log", "Ppt_rolling_168h_log",
            "hours_since_rain_log", "api_log", "DaySin", "DayCos"]
TARGET = "SWC_20_daily_avg"   # predict_avg: the calendar-day mean of SWC_20 at the target hour

ZOOM = ("2024-05-01", "2024-06-30")   # close-up period for the second plot
C_ACTUAL, C_MODEL, C_LAST = "#0b0b0b", "#2a78d6", "#eb6834"

## 1. Data and features

Every feature at hour *t* uses data at or before *t*, and is computed on the full `FD11` history so the 30-day
lookbacks are complete from the first modeling hour.

In [ ]:
if not os.path.exists(PARQUET_PATH):   # e.g. a Colab runtime, which cannot see the files on your computer
    print(f"{PARQUET_PATH} not found; downloading it from GitHub (84.5 MB) ...")
    urllib.request.urlretrieve(PARQUET_URL, PARQUET_PATH + ".part")
    with open(PARQUET_PATH + ".part", "rb") as f:
        assert hashlib.sha256(f.read()).hexdigest() == PARQUET_SHA256, "downloaded file does not match the dataset"
    os.replace(PARQUET_PATH + ".part", PARQUET_PATH)
raw = pd.read_parquet(PARQUET_PATH)
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"])
hist = raw[raw["Station"] == STATION].sort_values("Timestamp").set_index("Timestamp").asfreq("h")
assert not hist[["SWC_20", "SWC_5", "Ppt"]].isna().any().any(), "missing hours in the station history"

swc = hist["SWC_20"]
rain = hist["Ppt"].clip(lower=0.0)
rain = rain.mask(rain < PPT_DEADBAND_MM, 0.0)
day = hist.index.floor("D")

f = pd.DataFrame(index=hist.index)
f["SWC_20"] = swc
f["SWC_20_change_24h"] = swc - swc.shift(24)
f["SWC_20_change_72h"] = swc - swc.shift(72)
f["SWC_20_below_max_72h"] = swc - swc.rolling(72).max()     # <= 0: how far below the recent peak
f["SWC_20_above_min_30d"] = swc - swc.rolling(720).min()    # >= 0: water above the recent dry floor
f["SWC_5"] = hist["SWC_5"]
for hours in (24, 72, 168):
    f[f"Ppt_rolling_{hours}h_log"] = np.log1p(rain.rolling(hours).sum())
last_rain = pd.Series(hist.index.where(rain.to_numpy() > 0), index=hist.index).ffill()
hours_since = (hist.index.to_series() - last_rain).dt.total_seconds() / 3600
f["hours_since_rain_log"] = np.log1p(hours_since.fillna(720).clip(upper=720))
api, level = np.zeros(len(rain)), 0.0
for i, r in enumerate(rain.to_numpy()):   # antecedent precipitation index: decays 2% per hour
    level = 0.98 * level + r
    api[i] = level
f["api_log"] = np.log1p(api)
f["DaySin"] = np.sin(2 * np.pi * hist.index.hour / 24)
f["DayCos"] = np.cos(2 * np.pi * hist.index.hour / 24)
# Target: the daily mean, usable only if every hour of that day is an observed reading
f[TARGET] = swc.groupby(day).transform("mean")
f["target_observed"] = (hist["SWC_20_Origin"] == "observed").groupby(day).transform("all")

df = f[f.index.year.isin(TRAIN_YEARS + VAL_YEARS + TEST_YEARS)]
assert not df.isna().any().any()
train_df, val_df, test_df = (df[df.index.year.isin(years)] for years in (TRAIN_YEARS, VAL_YEARS, TEST_YEARS))
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert (np.diff(part.index.values) == np.timedelta64(1, "h")).all(), f"{name} is not consecutive hours"
    print(f"{name:5s} {part.index.min()} -> {part.index.max()}  {len(part)} hours")

## 2. Scaling and windows

A window is `WINDOW_SIZE` hours of inputs. Its target is the daily mean of `SWC_20` on the day of the hour that
falls `HORIZON` hours after the window's last hour, the same indexing as `model_test`'s `data_to_X_y`. The network
predicts the change from the window's last `SWC_20` value, and that value is added back for every score.

In [ ]:
scaler = MinMaxScaler().fit(train_df[FEATURES])               # training years only
SWC_RANGE = scaler.data_range_[FEATURES.index("SWC_20")]      # the change target uses SWC_20's scaled units


def make_windows(part, stride=1):
    """Windows of one partition whose target day is fully observed, then every `stride`-th one."""
    X_all = scaler.transform(part[FEATURES]).astype(np.float32)
    ends = np.arange(WINDOW_SIZE - 1, len(part) - HORIZON)           # last input hour of each window
    ends = ends[part["target_observed"].to_numpy(dtype=bool)[ends + HORIZON]][::stride]
    targets = ends + HORIZON
    swc_values = part["SWC_20"].to_numpy()
    last, y_true = swc_values[ends], part[TARGET].to_numpy()[targets]
    return {"X": np.stack([X_all[e - WINDOW_SIZE + 1:e + 1] for e in ends]),
            "y": ((y_true - last) / SWC_RANGE).astype(np.float32),   # scaled change since the last input hour
            "last": last, "y_true": y_true,
            "ma24": np.array([swc_values[e - 23:e + 1].mean() for e in ends]),
            "end_time": part.index[ends], "target_time": part.index[targets]}


train_w = make_windows(train_df, TRAINING_STRIDE)
val_w = make_windows(val_df, VALIDATION_STRIDE)
test_w = make_windows(test_df)                                 # every test window

for w in (train_w, val_w, test_w):
    assert ((w["target_time"] - w["end_time"]) == pd.Timedelta(hours=HORIZON)).all()
    assert (w["target_time"].floor("D") > w["end_time"]).all()  # the target day starts after the window ends
print(f"X {train_w['X'].shape} (windows, hours, features) | windows: train {len(train_w['y'])}, "
      f"val {len(val_w['y'])}, test {len(test_w['y'])}")

## 3. Model: `model_test`'s BiLSTM

In [ ]:
class BiLSTMModel(nn.Module):
    """model_test's BiLSTM: input projection, BiLSTM, LayerNorm and dropout, MLP decoder."""

    def __init__(self, input_dim, output_dim=1, num_layers=2, hidden_size=64, dropout=0.2):
        super().__init__()
        self.proj_inp = nn.Linear(input_dim, hidden_size)
        self.bilstm = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=num_layers,
                              batch_first=True, dropout=dropout if num_layers > 1 else 0, bidirectional=True)
        self.norm = nn.LayerNorm(2 * hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(nn.Linear(2 * hidden_size, 32), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(32, output_dim))

    def forward(self, x):
        x = self.proj_inp(x)                              # (batch, seq_len, hidden_size)
        _, (hn, _) = self.bilstm(x)                       # hn: (num_layers * 2, batch, hidden_size)
        x = torch.cat((hn[-2], hn[-1]), dim=1)            # final forward and backward states
        return self.fc(self.dropout(self.norm(x)))        # (batch, output_dim)

## 4. Training

In [ ]:
def loader(w, batch_size, shuffle):
    data = TensorDataset(torch.tensor(w["X"]), torch.tensor(w["y"]).unsqueeze(1))
    return DataLoader(data, batch_size=batch_size, shuffle=shuffle, generator=torch.Generator().manual_seed(SEED))


train_loader, val_loader = loader(train_w, BATCH_SIZE, True), loader(val_w, 1024, False)
torch.manual_seed(SEED)
model = BiLSTMModel(input_dim=len(FEATURES)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.HuberLoss(delta=HUBER_DELTA / SWC_RANGE)       # delta converted to scaled units

history = {"loss": [], "val_loss": []}
best_val, best_state, best_epoch, wait = float("inf"), None, 0, 0
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        val_loss = sum(criterion(model(xb.to(DEVICE)), yb.to(DEVICE)).item() * len(xb)
                       for xb, yb in val_loader) / len(val_loader.dataset)
    history["loss"].append(total / len(train_loader.dataset))
    history["val_loss"].append(val_loss)
    if val_loss < best_val:
        best_val, best_epoch, wait = val_loss, epoch + 1, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
        if wait >= PATIENCE:
            break
model.load_state_dict(best_state)                            # restore the best validation epoch
print(f"epochs run: {len(history['loss'])}, best epoch: {best_epoch}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history["loss"]) + 1), history["loss"], label="Train loss")
plt.plot(range(1, len(history["loss"]) + 1), history["val_loss"], label="Validation loss")
plt.axvline(best_epoch, color="gray", linestyle=":", label=f"best epoch ({best_epoch})")
plt.title("Loss curves")
plt.xlabel("Epoch")
plt.ylabel("Huber loss (scaled change)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Evaluation

Scores are in `SWC_20` units, on every validation and test window, against the baselines on the same targets.
Skill = 1 − model / last value; a positive value means the model beats persistence.

In [ ]:
def predict(w):
    model.eval()
    with torch.no_grad():
        out = torch.cat([model(torch.tensor(w["X"][i:i + 1024]).to(DEVICE)).cpu() for i in range(0, len(w["X"]), 1024)])
    return w["last"] + out.numpy().flatten() * SWC_RANGE       # add the predicted change back


def scores(y_true, y_pred):
    err = y_pred - y_true
    return {"RMSE": np.sqrt(np.mean(err ** 2)), "MAE": np.mean(np.abs(err)), "Bias": err.mean(),
            "CORR": np.corrcoef(y_true, y_pred)[0, 1]}


rows = {}
for name, w in [("val 2022", make_windows(val_df)), ("test 2024", test_w)]:
    pred = predict(w)
    if w is test_w:
        test_pred = pred
    last = scores(w["y_true"], w["last"])
    for label, p in [("BiLSTM", pred), ("Last value", w["last"]), ("Moving average (24h)", w["ma24"])]:
        s = scores(w["y_true"], p)
        rows[(name, label)] = {**s, "RMSE_skill": 1 - s["RMSE"] / last["RMSE"], "MAE_skill": 1 - s["MAE"] / last["MAE"]}
results = pd.DataFrame(rows).T
display(results.style.format("{:.5f}", subset=["RMSE", "MAE", "Bias", "CORR"])
        .format("{:+.1%}", subset=["RMSE_skill", "MAE_skill"]))

In [ ]:
t = pd.DataFrame({"actual": test_w["y_true"], "predicted": test_pred, "last_value": test_w["last"]},
                 index=test_w["target_time"])
zoom = t.loc[ZOOM[0]:ZOOM[1]]
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t.index, t["actual"], color=C_ACTUAL, linewidth=1.2, label="Original (daily mean)")
axes[0].plot(t.index, t["predicted"], color=C_MODEL, linewidth=1, linestyle="--", label="Predicted")
axes[0].set_title(f"BiLSTM predictions, test year 2024: SWC_20 daily mean, {HORIZON} h ahead")
axes[1].plot(zoom.index, zoom["actual"], color=C_ACTUAL, linewidth=1.4, label="Original (daily mean)")
axes[1].plot(zoom.index, zoom["predicted"], color=C_MODEL, linewidth=1.2, label="Predicted")
axes[1].plot(zoom.index, zoom["last_value"], color=C_LAST, linewidth=1, linestyle=":",
             label=f"Last value (SWC_20 {HORIZON} h earlier)")
axes[1].set_title(f"Close-up {ZOOM[0]} to {ZOOM[1]}")
for ax in axes:
    ax.set_ylabel("SWC_20")
    ax.legend()
    ax.grid(True)
plt.tight_layout()
plt.show()